# 05 — Le combinazioni con ModernBERT (tentativo 09)

Come la combinazione di `02_modelli`, con una famiglia in più: **ModernBERT** (`04_modernbert`), la media sui semi
finiti. Le famiglie testo (TF-IDF), link e pagina si ricalcolano qui con lo stesso codice di `02_modelli` (stesse
parti interne, stessi C scelti), perché lo stacking ha bisogno delle loro probabilità fuori fold. Si controlla che le
parti interne di ModernBERT siano esattamente le stesse.

Modelli: **ModernBERT** da solo; **combinato (ModernBERT+link+pagina)**, cioè il TF-IDF sostituito da ModernBERT;
**combinato (testo+ModernBERT)**; **combinato (tutte + ModernBERT)**. Risultati in
`Results/3-classificazione/modelli_bert/`, letti da `03_confronto`.

In [ ]:
import json
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from comune import (CLASSI, CRITERI, PUNTI, RESULTS, SEED, SEGNALI, VERSIONI, livello_da_punteggio, metriche_di, pesi_bilanciati,
                    registratore, testate)

In [ ]:
C_GRIGLIA = "0.01,0.1,1,10"
FAMIGLIE = "testo,link,pagina"
FOLDS = 0                   # 0 = tutti
PROVA = False               # True: ModernBERT finto da Results/3-classificazione/prova/modernbert_finto, solo per il codice

In [ ]:
OUT = RESULTS / ("prova" if PROVA else "") / "modelli_bert"
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT / "avanzamento.log")
T0 = time.time()
GRIGLIA = [float(c) for c in C_GRIGLIA.split(",")]
FAM = FAMIGLIE.split(",")
t = testate()
S = pd.read_csv(SEGNALI / "testate.csv.gz", index_col=0).reindex(t.index)
lnk = pd.read_csv(SEGNALI / "link_citati.csv.gz")
lnk = lnk[lnk.citato.map(t.gruppo).ne(lnk.domain.map(t.gruppo))]          # via le citazioni dentro il gruppo
art = pd.read_csv(VERSIONI / "dataset_finale.csv", usecols=["domain", "text"]); art["text"] = art.text.fillna("")
idx = {d: i for i, d in enumerate(t.index)}
A = sp.csr_matrix((np.ones(len(art)), (art.domain.map(idx).to_numpy(), np.arange(len(art)))), shape=(len(t), len(art)))
A = sp.diags(1 / np.maximum(A.sum(1).A1, 1)) @ A                          # media per testata
COL_LINK_TIPI = ["link citati per articolo"] + [c for c in S.columns if c.startswith("link: ")]
COL_PAGINA = [c for c in S.columns if c not in COL_LINK_TIPI and c not in ("link citati nel testo", "parole del testo estratto")]
y = t.label.to_numpy(); peso = t.peso.to_numpy(); val = t.valutazione.to_numpy(); folds = sorted(t.fold_gruppo.unique())
if FOLDS:
    folds = [int(FOLDS)]
if PROVA:
    GRIGLIA = GRIGLIA[1:3]
registra(f"=== {len(t)} testate ({val.sum()} valutate), {len(art)} articoli, {len(lnk)} citazioni; famiglie {FAM}; C {GRIGLIA} ===")

In [ ]:
def matrici(tr):
    tr = np.asarray(tr)
    M = {}
    if "testo" in FAM:
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True, max_features=100_000, strip_accents="unicode")
        art_tr = art.domain.map(idx).isin(tr).to_numpy()
        v.fit(art.text[art_tr])
        X = A @ v.transform(art.text)
        M["testo"] = (sp.diags(1 / np.maximum(sp.linalg.norm(X, axis=1), 1e-12)) @ X).tocsr()
    if "link" in FAM:
        cnt = lnk.groupby(["domain", "citato"]).volte.sum()
        presenti = set(cnt.index.get_level_values(0))
        righe = [cnt.loc[d].to_dict() if d in presenti else {} for d in t.index]
        dv = DictVectorizer(); D = dv.fit_transform(righe)
        tenuti = np.asarray((D[tr] > 0).sum(0)).ravel() >= 3                   # domini citati da almeno 3 testate del training
        D = D[:, tenuti]
        tf = TfidfTransformer(sublinear_tf=True).fit(D[tr])
        Dt = tf.transform(D)
        # affidabilità dei siti citati: solo testate di training (e mai dello stesso gruppo, già tolte)
        noto = set(t.index[tr])
        c = lnk[lnk.citato.isin(noto)]
        sc = c.citato.map(t.score)
        w = c.volte
        media = (sc * w).groupby(c.domain).sum() / w.groupby(c.domain).sum()
        inaff = ((sc < 60) * w).groupby(c.domain).sum() / w.groupby(c.domain).sum()
        n = w.groupby(c.domain).sum()
        V = pd.DataFrame({"punteggio medio dei siti citati": media, "quota citazioni verso inaffidabili": inaff,
                          "citazioni di testate note (log)": np.log1p(n)}).reindex(t.index)
        V["nessuna citazione di testate note"] = V["punteggio medio dei siti citati"].isna().astype(float)
        dense = pd.concat([S[COL_LINK_TIPI], V], axis=1)
        dense = dense.fillna(dense.iloc[tr].mean()).fillna(0)
        sc_ = StandardScaler().fit(dense.iloc[tr])
        M["link"] = sp.hstack([Dt, sp.csr_matrix(sc_.transform(dense))]).tocsr()
        M["_nomi_link"] = list(np.asarray(dv.get_feature_names_out())[tenuti]) + list(dense.columns)
    if "pagina" in FAM:
        P = S[COL_PAGINA].fillna(S[COL_PAGINA].iloc[tr].mean()).fillna(0)
        sc_ = StandardScaler().fit(P.iloc[tr])
        M["pagina"] = sp.csr_matrix(sc_.transform(P))
        M["_nomi_pagina"] = list(P.columns)
    return M

In [ ]:
def adatta(X, yy, ww, c, binario=False):
    pw = pesi_bilanciati(yy, ww)
    return LogisticRegression(C=c, max_iter=5000, random_state=SEED).fit(X, yy, sample_weight=pw)


def proba(m, X, k):
    P = np.zeros((X.shape[0], k)); P[:, m.classes_.astype(int)] = m.predict_proba(X)
    return P


def punteggio(yv, Pv, binario):
    if binario:
        return balanced_accuracy_score(yv, (Pv[:, 1] >= 0.5).astype(int))
    return f1_score(yv, Pv.argmax(1), average="macro", labels=range(5), zero_division=0)


def interni(tr):
    # 3 fold per gruppi sulle testate di training: (indici di training, indici di validazione, matrici)
    tt = t.iloc[tr]
    out = []
    for a, b in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(tt, tt.label, tt.gruppo):
        out.append((tr[a], tr[b], matrici(tr[a])))
    return out


def famiglia(fam, tr, te, M, INT, yy, k, binario):
    # sceglie C nella validazione interna; ritorna probabilità sul test e fuori fold sul training
    noti = ~np.isnan(yy)
    punti, oof = {}, {}
    for c in GRIGLIA:
        s, P = [], np.full((len(t), k), np.nan)
        for a, b, Mi in INT:
            a, b2 = a[noti[a]], b[noti[b]]
            m = adatta(Mi[fam][a], yy[a].astype(int), peso[a], c, binario)
            P[b2] = proba(m, Mi[fam][b2], k)
            bv = b2[val[b2]]
            s.append(punteggio(yy[bv].astype(int), P[bv], binario))
        punti[c], oof[c] = float(np.mean(s)), P
    C = max(punti, key=punti.get)
    tr2 = tr[noti[tr]]
    m = adatta(M[fam][tr2], yy[tr2].astype(int), peso[tr2], C, binario)
    return proba(m, M[fam][te], k), oof[C][tr], C, punti, m

## ModernBERT: la media sui semi finiti

In [ ]:
CARTELLA_BERT = RESULTS / ("prova/modernbert_finto" if PROVA else "modernbert")
SEMI = sorted(int(p.name.split("_s")[1]) for p in CARTELLA_BERT.glob("modernbert_s*") if (p / "FATTO").exists() or PROVA)
assert SEMI, "nessun seme di ModernBERT finito"


def bert(nome):
    tabs = [pd.read_csv(CARTELLA_BERT / f"modernbert_s{s}" / f"{nome}.csv").set_index("domain")[CLASSI] for s in SEMI]
    return sum(tabs) / len(tabs)


registra(f"ModernBERT: semi {SEMI}")

## Tutti i fold

In [ ]:
def combina(oof_fam, te_fam, yy, tr, k):
    # come in 02_modelli: meta-modello sulle probabilità fuori fold delle famiglie (log), addestrato sul solo training
    noti = ~np.isnan(yy[tr])
    Z = np.hstack([np.log(np.clip(oof_fam[f], 1e-6, 1)) for f in oof_fam])[noti]
    Zt = np.hstack([np.log(np.clip(te_fam[f], 1e-6, 1)) for f in te_fam])
    m = adatta(Z, yy[tr][noti].astype(int), peso[tr][noti], 1.0)
    return proba(m, Zt, k)


COMBO = {"combinato (ModernBERT+link+pagina)": ["bert", "link", "pagina"], "combinato (testo+ModernBERT)": ["testo", "bert"],
         "combinato (tutte + ModernBERT)": ["testo", "bert", "link", "pagina"]}
COMBO = {n: fs for n, fs in COMBO.items() if all(x == "bert" or x in FAM for x in fs)}
PROB = defaultdict(lambda: np.full((len(t), 5), np.nan))
for f in folds:
    t1 = time.time()
    tr, te = np.where(t.fold_gruppo != f)[0], np.where(t.fold_gruppo == f)[0]
    M = matrici(tr); INT = interni(tr)
    te_p, oof_p = {}, {}
    for fam in FAM:
        te_p[fam], oof_p[fam], C, punti, _ = famiglia(fam, tr, te, M, INT, y.astype(float), 5, False)
    O = np.full((len(t), 5), np.nan)
    for k, (a, b, _) in enumerate(INT, 1):
        pk = bert(f"fold{f}_interno{k}")
        assert set(pk.index) == set(t.index[b]), f"fold {f} parte {k}: le testate di ModernBERT non sono quelle della parte interna"
        O[b] = pk.reindex(t.index[b]).to_numpy()
    T = np.full((len(t), 5), 0.2)                     # le testate di solo training del test non si valutano
    tv = te[val[te]]
    pf = bert(f"fold{f}_finale")
    assert set(pf.index) == set(t.index[tv]), f"fold {f}: le testate di test di ModernBERT non sono quelle valutate"
    T[tv] = pf.reindex(t.index[tv]).to_numpy()
    oof_p["bert"], te_p["bert"] = O[tr], T[te]
    assert not np.isnan(oof_p["bert"]).any()
    PROB["ModernBERT"][te] = te_p["bert"]
    for nome, fs in COMBO.items():
        PROB[nome][te] = combina({x: oof_p[x] for x in fs}, {x: te_p[x] for x in fs}, y.astype(float), tr, 5)
    righe_log = []
    for nome in ["ModernBERT", *COMBO]:
        mm = metriche_di(y[tv], PROB[nome][tv].argmax(1)); righe_log.append(f"{nome} {mm['f1_macro']:.3f}/{mm['bin_accuratezza']:.3f}")
    registra(f"fold {f} ({(time.time() - t1) / 60:.1f} min), F1/binario: " + ", ".join(righe_log))

In [ ]:
fatti = ~np.isnan(PROB["ModernBERT"][:, 0])
righe = pd.DataFrame({"domain": t.index, "fold": t.fold_gruppo, "valutazione": val, "y_true": y})
for nome, P in PROB.items():
    for k in range(5):
        righe[f"{nome} | p_l{k + 1}"] = P[:, k]
righe[fatti].to_csv(OUT / "predizioni_testate.csv", index=False)
v = fatti & val
tab = {nome: {k: m for k, m in metriche_di(y[v], P[v].argmax(1)).items() if not isinstance(m, list)} for nome, P in PROB.items()}
tab = pd.DataFrame(tab).T[["n", "f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza", "bin_f1_macro"]]
tab.to_csv(OUT / "metriche.csv")
(OUT / "semi.txt").write_text(" ".join(map(str, SEMI)))
registra(f"=== finito in {(time.time() - T0) / 60:.1f} min, semi {SEMI} ===\n" + tab.round(3).to_string())
tab.round(3)